# Affordance

> Extract minimal affordances (shape, rule, ontology) from any RDF/JSON‑LD so the LLM knows *what* operations it could attempt.

In [ ]:
#| default_exp reason.afford

In [ ]:
#| export
from __future__ import annotations
import json
from typing import List
from pydantic import BaseModel
from rdflib import Graph, RDF
from cogitarelink.core.debug import get_logger

In [ ]:
#| export
log = get_logger("afford")


In [ ]:
#| export
class Affordance(BaseModel):
    iri:str
    kind:str          # "shape" | "rule" | "ontology"
    source:str="inline"

def _to_graph(data:str|dict) -> Graph:
    g=Graph()
    if isinstance(data,str): g.parse(data=data, format="json-ld")
    else:                    g.parse(data=json.dumps(data), format="json-ld")
    return g

class AffordanceScanner:
    HINTS = {
        "NodeShape":"shape",
        "SPARQLRule":"rule",
        "Ontology":"ontology",
    }
    def scan(self, data:str|dict) -> List[Affordance]:
        g=_to_graph(data)
        out=[]
        for s,_,o in g.triples((None,RDF.type,None)):
            term = str(o).split('#')[-1]
            if term in self.HINTS:
                out.append(Affordance(iri=str(s), kind=self.HINTS[term]))
        log.debug("found %d affordances", len(out))
        return out

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()